In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
from delta.tables import DeltaTable
from datetime import datetime
from pyspark.sql.window import Window

In [0]:
# =====================================================
# SILVER LAYER - FHIR Patient Processing (Production Ready)
# =====================================================

from pyspark.sql.types import *
from pyspark.sql.functions import *
from datetime import datetime
import logging

# ====================== Config Details ======================
CONFIG = {
    "resource": "Patient",
    "run_date": datetime.today().strftime("%Y-%m-%d"),
    "bronze_path": "fhir_data.prd_bronze",
    "silver_path": "fhir_data.prd_silver"
}

resource = CONFIG["resource"]
run_date = CONFIG["run_date"]
bronze_base = CONFIG["bronze_path"]
silver_base = CONFIG["silver_path"]

# ====================== LOGGING ======================
logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')
logger = logging.getLogger(__name__)

logger.info(f"Starting Silver Layer | Resource: {resource} | Run Date: {run_date}")

# ====================== SCHEMA ======================
schema_patient = StructType([
    StructField("active", BooleanType()),
    StructField("birthDate", StringType()),
    StructField("gender", StringType()),
    StructField("id", StringType()),
    StructField("resourceType", StringType()),
    StructField("meta", StructType([
        StructField("lastUpdated", StringType()),
        StructField("source", StringType()),
        StructField("versionId", StringType())
    ])),
    StructField("name", ArrayType(StructType([
        StructField("family", StringType()),
        StructField("given", ArrayType(StringType())),
        StructField("use", StringType())
    ]))),
    StructField("telecom", ArrayType(StructType([
        StructField("system", StringType()),
        StructField("value", StringType())
    ])))
])

# ====================== MAIN FUNCTION ======================
def process_patient_silver(resource, run_date, bronze_base):

    bronze_path = f"{bronze_base}.{resource.lower()}"

    logger.info(f"Reading bronze data from: {bronze_path}")

    df = spark.read\
        .table(bronze_path) \
        .filter(col("load_date") == run_date)

    logger.info(f"Initial count: {df.count()}")

    df = df.dropDuplicates()

    # ================= PARSE =================
    df_parsed = df.withColumn("parsed", from_json(col("resource"), schema_patient))

    # ================= FLATTEN =================
    df_flat = df_parsed.select(
        col("parsed.id").alias("resource_id"),
        col("parsed.gender").alias("gender"),
        col("parsed.birthDate").alias("birth_date_raw"),
        col("parsed.meta.lastUpdated").alias("last_updated_raw"),
        col("parsed.meta.source").alias("source"),
        col("parsed.resourceType").alias("resource_type"),
        col("parsed.meta.versionId").alias("version_id_raw"),
        col("parsed.active").alias("active_raw"),
        explode_outer(col("parsed.name")).alias("name"),
        col("parsed.telecom").alias("telecom_array")
    )

    logger.info(f"After flatten: {df_flat.count()}")

    # ================= NAME =================
    df_names = df_flat \
        .withColumn("family_name_raw", col("name.family")) \
        .withColumn("given_name_raw", concat_ws(" ", col("name.given"))) \
        .withColumn("name_use", col("name.use")) \
        .drop("name")

    # ================= CONTACT CLEANING =================
    df_contacts = df_names \
        .withColumn(
            "phone_number",
            expr("""
                CASE 
                    WHEN telecom_array IS NULL THEN NULL
                    ELSE
                        CASE 
                            WHEN size(
                                filter(
                                    transform(
                                        telecom_array,
                                        x -> regexp_replace(x.value, '[^0-9+]', '')
                                    ),
                                    x -> x IS NOT NULL AND length(x) >= 10
                                )
                            ) = 0 THEN NULL
                            ELSE concat_ws(', ',
                                filter(
                                    transform(
                                        telecom_array,
                                        x -> regexp_replace(x.value, '[^0-9+]', '')
                                    ),
                                    x -> x IS NOT NULL AND length(x) >= 10
                                )
                            )
                        END
                END
            """)
        ) \
        .withColumn(
            "email_address",
            expr("""
                CASE 
                    WHEN size(
                        filter(telecom_array, 
                            x -> x.system = 'email' 
                            AND x.value RLIKE '^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\\.[A-Za-z]{2,}$'
                        )
                    ) = 0 THEN NULL
                    ELSE concat_ws(', ',
                        transform(
                            filter(telecom_array, 
                                x -> x.system = 'email' 
                                AND x.value RLIKE '^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\\.[A-Za-z]{2,}$'
                            ),
                            x -> x.value
                        )
                    )
                END
            """)
        ) \
        .drop("telecom_array")

    # ================= TRANSFORM =================
    df_transformed = df_contacts \
        .withColumn("birth_date", to_date(col("birth_date_raw"), "yyyy-MM-dd")) \
        .withColumn("last_updated",to_timestamp(col("last_updated_raw"))) \
        .withColumn("version_id", col("version_id_raw").cast("string")) \
        .withColumn("is_active", col("active_raw").cast("boolean")) \
        .withColumn("family_name", initcap(trim(col("family_name_raw")))) \
        .withColumn("given_name", initcap(trim(col("given_name_raw")))) \
        .withColumn("gender", lower(col("gender")))

    # ================= DERIVED =================
    #.filter((col("family_name").rlike("^\\.$")) &(~col("family_name").rlike("(?i)citacell-test")))
    df_transformed = df_transformed \
        .withColumn("last_name_clean",expr("""CASE WHEN length(family_name) % 2 = 0 AND substring(family_name, 1, length(family_name)/2) = substring(family_name, length(family_name)/2 + 1, length(family_name)/2)
        THEN substring(family_name, 1, length(family_name)/2)
        ELSE family_name
    END
    """)
)\
        .withColumn("age", (datediff(current_date(), col("birth_date")) / 365.25).cast("int")) \
        .withColumn("age_group",
            when(col("age") < 18, "Minor")
            .when(col("age").between(18, 35), "Young Adult")
            .when(col("age").between(36, 55), "Middle Age")
            .when(col("age").between(56, 65), "Pre-Senior")
            .when(col("age") > 65, "Senior")
            .otherwise("Unknown")
        )

    # ================= DATA QUALITY =================
    df_clean = df_transformed.filter(
        col("resource_id").isNotNull() &
        col("birth_date").isNotNull() &
        col("gender").isNotNull()
    )

    logger.info(f"Clean records count: {df_clean.count()}")

    # ================= FINAL SELECT =================
    df_final = df_clean.select(
        col("resource_id").alias("Resource_id"),
        col("gender").alias("Gender"),
        col("birth_date").alias("Birth_Date"),
        col("last_updated").alias("Last_updated"),
        col("source").alias("Source"),
        col("resource_type").alias("Resource_Type"),
        col("version_id").alias("Version_id"),
        col("is_active").alias("Active"),
        col("last_name_clean").alias("Last_name"),
        col("given_name").alias("First_name"),
        col("name_use").alias("Name_use"),
        col("phone_number").alias("Phone_Number"),
        col("email_address").alias("Email"),
        col("age").alias("Age"),
        col("age_group").alias("Age_Group")
    ).withColumn("load_date", to_date(lit(run_date))) \
     .withColumn("ingestion_timestamp", current_timestamp())\
         .withColumn("effective_start_date", current_timestamp()) \
                     .withColumn("effective_end_date", lit(None).cast("timestamp")) \
                     .withColumn("is_current", lit(True))
    window_spec = Window.partitionBy("resource_id") \
                        .orderBy(col("last_updated").desc())

    df_final = (
        df_clean\
        .withColumn("rn", row_number().over(window_spec))\
        .filter(col("rn") == 1)\
        .drop("rn"))
    # df_final.display()
    # df_final.printSchema()

    return df_final


# ====================== EXECUTION ======================
try:
    df_silver = process_patient_silver(resource, run_date, bronze_base)
    
    df_silver.createOrReplaceTempView("src_enc")

    silver_path = f"{silver_base}.{resource.lower()}"

    spark.sql(f"""
MERGE INTO {silver_path} t
USING src_enc s
ON t.resource_id = s.resource_id AND t.is_current = true

-- 1. Expire old record (SCD Type 2)
WHEN MATCHED AND (
    t.gender <> s.gender OR
    t.birth_date <> s.birth_date OR
    t.last_updated <> s.last_updated OR
    t.source <> s.source OR
    t.resource_type <> s.resource_type OR
    t.version_id <> s.version_id OR
    t.active <> s.is_active OR
    t.last_name <> s.last_name_clean OR
    t.first_name <> s.given_name OR
    t.name_use <> s.name_use OR
    t.phone_number <> s.phone_number OR
    t.email <> s.email_address OR
    t.age <> s.age OR
    t.age_group <> s.age_group
)
THEN UPDATE SET
    t.is_current = false,
    t.effective_end_date = current_timestamp()

-- 2. Insert new record
WHEN NOT MATCHED
THEN INSERT (
    resource_id,
    gender,
    birth_date,
    last_updated,
    source,
    resource_type,
    version_id,
    active,
    last_name,
    first_name,
    name_use,
    phone_number,
    email,
    age,
    age_group,
    load_date,
    ingestion_timestamp,
    effective_start_date,
    effective_end_date,
    is_current
)
VALUES (
    s.resource_id,
    s.gender,
    s.birth_date,
    s.last_updated,
    s.source,
    s.resource_type,
    s.version_id,
    s.is_active,
    s.last_name_clean,
    s.given_name,
    s.name_use,
    s.phone_number,
    s.email_address,
    s.age,
    s.age_group,
    current_date(),          
    current_timestamp(), 
    current_timestamp(),
    NULL,
    true
)
""")
    # df_silver.write.format("delta") \
    #     .mode("append") \
    #     .option("mergeSchema", "true") \
    #     .saveAsTable(silver_path)

    logger.info(f"Successfully written to Silver: {silver_path}")

    #df_silver.display()
    df_silver.printSchema()

except Exception as e:
    logger.error(f"Pipeline failed: {str(e)}", exc_info=True)
    raise

finally:
    logger.info("Pipeline execution completed")